# 04 — Model Merge & Export

**Goal**: Learn the end-to-end workflow for merging LoRA adapters, saving in standard formats, quantizing, and pushing to the Hugging Face Hub.

**Prerequisites**: Stage 1 notebooks 01-03 (you should have a trained LoRA adapter).

In [ ]:
!pip install peft transformers huggingface_hub auto-gptq accelerate torch -q

## 1. Load Base Model + LoRA Adapter

In production, LoRA adapters are small weight deltas stored separately. Before deployment, you typically merge them back into the base model so there is no adapter-dispatch overhead at inference time.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
ADAPTER_PATH = "./dpo-lora-adapter"  # or "./qwen-lora-adapter"

# Load base model
print(f"Loading base model: {MODEL_NAME}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA adapter
try:
    config = PeftConfig.from_pretrained(ADAPTER_PATH)
    print(f"LoRA config: r={config.r}, alpha={config.lora_alpha}, target_modules={config.target_modules}")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    print("Adapter loaded successfully.")
except Exception as e:
    print(f"No adapter found at {ADAPTER_PATH}. Using base model for demo.\n({e})")
    model = base_model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model ready. Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 2. Merge LoRA Weights with `merge_and_unload()`

This fuses the LoRA matrices into the base weights, producing a standard (non-PEFT) model. After merging, the model is a single set of weights -- no adapter loading needed at inference.

In [ ]:
print("Before merge:")
print(f"  Type: {type(model).__name__}")
print(f"  Has LoRA adapter: {hasattr(model, 'peft_config')}")

# Merge LoRA weights into base model, then discard the adapter wrapper
merged_model = model.merge_and_unload()

print("\nAfter merge:")
print(f"  Type: {type(merged_model).__name__}")
print(f"  Has LoRA adapter: {hasattr(merged_model, 'peft_config')}")
print(f"  Parameters: {sum(p.numel() for p in merged_model.parameters()) / 1e9:.2f}B")
print("\nNote: Parameter count should match the base model (LoRA weights have been absorbed).")

## 3. Save the Merged Model in Hugging Face Format

The standard format is:
```
merged-model/
  config.json           # model architecture config
  model.safetensors     # weights (safe format, recommended)
  tokenizer.json
  tokenizer_config.json
  special_tokens_map.json
```

Use `safetensors` (not `.bin`) -- they are faster, safer (no pickle exploits), and the HF standard.

In [ ]:
import os

MERGE_OUTPUT = "./my-finetuned-model"
os.makedirs(MERGE_OUTPUT, exist_ok=True)

# Save merged model in safetensors format
merged_model.save_pretrained(MERGE_OUTPUT, safe_serialization=True)
tokenizer.save_pretrained(MERGE_OUTPUT)

# Verify saved files
saved_files = os.listdir(MERGE_OUTPUT)
print(f"Files saved to '{MERGE_OUTPUT}':")
for f in sorted(saved_files):
    fpath = os.path.join(MERGE_OUTPUT, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f} ({size_mb:.1f} MB)")

total_mb = sum(os.path.getsize(os.path.join(MERGE_OUTPUT, f)) for f in saved_files) / (1024**2)
print(f"\nTotal size: {total_mb:.1f} MB")

## 4. Reload the Merged Model & Sanity Check

Always verify that the merged model loads cleanly and generates reasonable outputs before shipping.

In [ ]:
# Reload the merged model
reloaded = AutoModelForCausalLM.from_pretrained(
    MERGE_OUTPUT,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
reloaded_tokenizer = AutoTokenizer.from_pretrained(MERGE_OUTPUT, trust_remote_code=True)
if reloaded_tokenizer.pad_token is None:
    reloaded_tokenizer.pad_token = reloaded_tokenizer.eos_token

# Generate a test output
test_prompt = "How do I reset my password?"
messages = [{"role": "user", "content": test_prompt}]
text = reloaded_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = reloaded_tokenizer(text, return_tensors="pt").to(reloaded.device)

with torch.no_grad():
    outputs = reloaded.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=reloaded_tokenizer.eos_token_id,
    )
response = reloaded_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print(f"Prompt: {test_prompt}")
print(f"Response: {response}")
print("\nMerged model loads and generates correctly.")

## 5. Push to Hugging Face Hub

Prerequisites:
1. Create a HF account at huggingface.co
2. Run `huggingface-cli login` and paste your access token (from Settings > Access Tokens)
3. Or set `HF_TOKEN` environment variable

In [ ]:
from huggingface_hub import HfApi, create_repo, upload_folder

# Configuration -- replace with your values
HF_USERNAME = "your-username"        # TODO: change this
HF_REPO_NAME = "my-finetuned-qwen2.5-cs"
REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

print(f"Target repo: https://huggingface.co/{REPO_ID}")
print("\nTo push, run:")
print(f"  1. huggingface-cli login")
print(f"  2. create_repo('{REPO_ID}', private=True, exist_ok=True)")
print(f"  3. upload_folder(folder_path='{MERGE_OUTPUT}', repo_id='{REPO_ID}')")

# --- Uncomment below to actually push ---
# create_repo(REPO_ID, private=True, exist_ok=True)
# upload_folder(
#     folder_path=MERGE_OUTPUT,
#     repo_id=REPO_ID,
#     commit_message="Initial upload: merged Qwen2.5-0.5B after DPO",
# )
# print(f"Model pushed to https://huggingface.co/{REPO_ID}")

## 6. GGUF Export (for llama.cpp / Ollama)

GGUF is the format used by llama.cpp and Ollama for CPU-efficient inference. The conversion is a CLI process, shown here as a markdown guide.

**Step-by-step:**

```bash
# 1. Clone llama.cpp
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp && make

# 2. Install Python dependencies
pip install -r requirements.txt

# 3. Convert HF model to GGUF
python convert_hf_to_gguf.py ../my-finetuned-model \
    --outfile my-model-Q8_0.gguf \
    --outtype q8_0

# 4. (Optional) Quantize further
./build/bin/llama-quantize my-model-Q8_0.gguf \
    my-model-Q4_K_M.gguf Q4_K_M

# 5. Run with llama.cpp
./build/bin/llama-cli -m my-model-Q4_K_M.gguf -p "Hello, how are you?"
```

**For Ollama users:**
1. Create a `Modelfile`:
   ```
   FROM ./my-model-Q4_K_M.gguf
   TEMPLATE """{{ .System }}\nUser: {{ .Prompt }}\nAssistant: """
   ```
2. `ollama create my-model -f Modelfile`
3. `ollama run my-model`

## 7. GPTQ Quantization with AutoGPTQ

GPTQ is a post-training quantization technique that compresses weights to 4/3/2 bits with minimal accuracy loss. It is the preferred GPU-deployment quantization.

In [ ]:
from transformers import AutoTokenizer
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

# For AutoGPTQ, we need calibration data -- a few hundred tokens of representative text
calibration_texts = [
    "How do I reset my password? To reset your password, go to Settings.",
    "My order hasn't arrived yet. I understand your concern.",
    "Do you ship internationally? Yes, we ship to over 60 countries!",
    "What is your return policy? We offer a 30-day return window.",
    "How do I cancel my subscription? To cancel, go to Account > Subscriptions.",
    "The app keeps crashing. Sorry about that! Let's try a few things.",
    "Do you have a student discount? Yes! Students get 20% off.",
    "I was charged twice for the same order. I apologize for the double charge.",
] * 16  # repeat to get enough calibration tokens

# Set up quantization config
quantize_config = BaseQuantizeConfig(
    bits=4,                  # 4-bit quantization
    group_size=128,          # group size for quantization
    damp_percent=0.01,
    desc_act=False,          # don't quantize activations for faster inference
    sym=True,                # symmetric quantization
    true_sequential=True,
)

# Quantize
print("Quantizing model to 4-bit GPTQ...")
gptq_model = AutoGPTQForCausalLM.from_pretrained(
    MERGE_OUTPUT,
    quantize_config=quantize_config,
    trust_remote_code=True,
)

gptq_model.quantize(calibration_texts, use_triton=False)

# Save quantized model
GPTQ_OUTPUT = "./my-model-gptq-4bit"
gptq_model.save_quantized(GPTQ_OUTPUT, use_safetensors=True)
tokenizer.save_pretrained(GPTQ_OUTPUT)

print(f"GPTQ model saved to '{GPTQ_OUTPUT}'")

# Compare sizes
original_size = sum(os.path.getsize(os.path.join(MERGE_OUTPUT, f)) for f in os.listdir(MERGE_OUTPUT)) / 1024**2
quantized_size = sum(os.path.getsize(os.path.join(GPTQ_OUTPUT, f)) for f in os.listdir(GPTQ_OUTPUT)) / 1024**2
print(f"\nOriginal: {original_size:.1f} MB")
print(f"GPTQ 4-bit: {quantized_size:.1f} MB")
print(f"Compression: {original_size / quantized_size:.1f}x")

## 8. Version Management & Directory Structure

A clean directory layout saves debugging time. Here is a recommended structure for a production fine-tuning project:

```
project-root/
  data/
    raw/                  # original datasets
    processed/            # tokenized & formatted
    preference_pairs.json

  checkpoints/
    sft-v1/               # SFT checkpoint
    dpo-v1-beta0.1/       # DPO checkpoints by hyperparams
    dpo-v2-beta0.05/

  adapters/
    sft-lora/             # LoRA weights only (small!)
    dpo-lora/

  exports/
    merged-fp16/          # merged model, full precision
    gptq-4bit/            # GPTQ quantized
    gguf/                 # GGUF files for llama.cpp

  eval/
    baseline-metrics.json
    v2-benchmark-results.json

  configs/
    sft_config.yaml
    dpo_config.yaml

  model_card.md           # HF model card
  README.md
```

**Version naming convention**: `{stage}-{version}-{key-hyperparams}`, e.g. `dpo-v2-beta0.05-lr1e5`

## 9. Incremental Fine-Tuning Workflow

This is the complete workflow from base model to deployment-ready artifact:

```
Base Model (Qwen2.5-7B)
  |
  |-- Step 1: SFT on domain data (notebook 01)
  |   Output: sft-lora-adapter/
  |
  |-- Step 2: Load base + SFT adapter, merge (this notebook)
  |   Output: merged-sft/
  |
  |-- Step 3: DPO on preference data (notebook 03)
  |   Reference = merged-sft/  (NOT the raw base model!)
  |   Output: dpo-lora-adapter/
  |
  |-- Step 4: Load merged-sft + DPO adapter, merge & unload
  |   Output: merged-dpo/
  |
  |-- Step 5: Quantize
  |   GPTQ for GPU serving / GGUF for CPU/edge
  |   Output: model-gptq-4bit/ or model.Q4_K_M.gguf
  |
  |-- Step 6: Push to Hub & Deploy
```

**Critical rule**: Always save the LoRA adapter separately BEFORE merging. Merging is one-way (destructive for the adapter). Keep adapters versioned.

In [ ]:
# Pseudocode for the complete incremental workflow
print("=" * 60)
print("COMPLETE WORKFLOW EXAMPLE (Pseudocode)")
print("=" * 60)

print("""
from transformers import AutoModelForCausalLM
from peft import PeftModel

# --- Phase 1: SFT ---
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B", torch_dtype=torch.float16)
# ... train SFT with LoRA ...
# sft_model.save_pretrained("./adapters/sft-lora")

# --- Phase 2: Merge SFT ---
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B", torch_dtype=torch.float16)
sft = PeftModel.from_pretrained(base, "./adapters/sft-lora")
merged_sft = sft.merge_and_unload()
merged_sft.save_pretrained("./checkpoints/sft-v1")

# --- Phase 3: DPO (reference = SFT checkpoint) ---
base_sft = AutoModelForCausalLM.from_pretrained("./checkpoints/sft-v1", torch_dtype=torch.float16)
# ... train DPO with LoRA, using base_sft as reference ...
# dpo_model.save_pretrained("./adapters/dpo-lora")

# --- Phase 4: Merge DPO ---
dpo = PeftModel.from_pretrained(base_sft, "./adapters/dpo-lora")
merged_dpo = dpo.merge_and_unload()
merged_dpo.save_pretrained("./checkpoints/dpo-v1")

# --- Phase 5: Quantize & Deploy ---
# GPTQ or GGUF on ./checkpoints/dpo-v1
""")

## 10. Quantization Quick Reference

| Method | Bits | File Size (7B) | Speed | Quality | Best For |
|--------|------|----------------|-------|---------|----------|
| FP16 | 16 | ~14 GB | baseline | perfect | Training, evaluation |
| INT8 | 8 | ~7 GB | 1.3x | near-perfect | CPU inference |
| GPTQ 4-bit | 4 | ~4 GB | 2-3x | very good | GPU server |
| AWQ 4-bit | 4 | ~4 GB | 2-3x | very good | GPU server (faster than GPTQ) |
| GGUF Q4_K_M | 4 | ~4 GB | 1.5x | good | CPU / edge / Ollama |
| GGUF Q2_K | 2 | ~2.5 GB | 2x | degraded | Extreme compression |

**Rule of thumb**: 4-bit is the sweet spot -- half the size of 8-bit with negligible quality loss for most tasks.

## TODO Exercises

1. **Merge your SFT adapter**: If you completed notebook 01, merge your SFT LoRA adapter and verify the merged model generates the same outputs as the adapter-loaded model.
2. **Push to HF Hub**: Create a Hugging Face account, generate an access token, and push your merged model.
3. **Try GGUF conversion**: Install llama.cpp and convert your merged model to GGUF format. Benchmark token/s on CPU.
4. **Compare quantization methods**: If you have a GPU, quantize the same model with GPTQ and AWQ. Compare inference speed and quality on 10 test prompts.
5. **Build an automated export script**: Write a Python script that takes a LoRA adapter path and automates: merge -> save -> quantize -> push to Hub.
6. **Model card**: Write a comprehensive model card (`model_card.md`) for your fine-tuned model using the HF template.

## Summary

- `merge_and_unload()` fuses LoRA weights into the base model, producing a standard model without PEFT dependency.
- Always save LoRA adapters separately before merging -- merging is one-way.
- `.safetensors` is the recommended serialization format (safe, fast, no pickle).
- `huggingface_hub` provides `create_repo()` + `upload_folder()` for push-to-Hub workflows.
- GGUF (via llama.cpp) is for CPU/edge/Ollama; GPTQ/AWQ are for GPU server deployments.
- 4-bit quantization is the sweet spot for most deployment scenarios.
- Keep a clean versioned directory structure -- you will thank yourself later.
- This concludes **Stage 1: Fine-tuning**. You now have the complete workflow: SFT -> LoRA -> DPO -> Merge -> Quantize -> Deploy.